# Spike Data Playground
This notebook achieves the following:
1. Download real-world raw unlabeled MEA data from the International Brain Lab using an API
2. Preprocessing data
3. Simulate raw synthetic MEA data
3. Working with data as a matrix


Note: The [SpikeInterface](https://spikeinterface.readthedocs.io/en/stable) package conveniently allows us to do all of the above. However, for the purposes of organization, we will download data separately. Additionally, SpikeInterface has a conveninent [pipeline setup](https://spikeinterface.readthedocs.io/en/stable/how_to/build_pipeline_with_dicts.html) we will utilize in the final code for downloading data, creating synthetic data, and preprocessing data.

Note: Neural recordings are usually very large (30+ GBs). Unfortunately, we have to download entire file (even if we are intereted in first few minutes of recording).

In [2]:
# update pip, then auto-install any missing packages by PyPI name
import sys, subprocess, re
from importlib import metadata as im

pkgs = [
    'ipykernel', 'numpy', 'pandas', 'matplotlib', 'scipy',
    'ONE-api', 'spikeinterface[full]', 'mtscomp',
    'kilosort', 'spython'
]

def _pip(*args):
    return subprocess.run([sys.executable, '-m', 'pip', *args], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode == 0

def _has(dist_name: str) -> bool:
    try:
        im.version(dist_name)  # checks distribution by PyPI/project name
        return True
    except im.PackageNotFoundError:
        return False

# 1) Ensure pip is current
_pip('install', '--upgrade', 'pip', '--quiet')

# 2) Ensure each package (extras like [full] are fine for install; strip for presence check)
report = {}
for spec in pkgs:
    dist = re.split(r'[\[\]]', spec)[0]          # e.g., 'spikeinterface[full]' -> 'spikeinterface'
    if _has(dist):
        report[spec] = 'already installed'
    else:
        ok = _pip('install', spec, '--quiet')
        report[spec] = 'installed' if ok and _has(dist) else 'install failed'

# 3) Summary
w = max(len(k) for k in report) + 2
for k, v in report.items(): print(f"{k.ljust(w)}{v}")

ipykernel             already installed
numpy                 already installed
pandas                already installed
matplotlib            already installed
scipy                 already installed
ONE-api               already installed
spikeinterface[full]  already installed
mtscomp               already installed
kilosort              already installed
spython               already installed


In [ ]:
# Assure matplotlib has white background
import matplotlib.pyplot as plt
plt.style.use("default")

# Real World Data
One source for multielectrode extracellular array (MEA) data is the the International Brain Lab ([IBL](https://www.internationalbrainlab.com/data)). It is essentially a collection of labs from across the world that collect data in some standardized format (ie., all labs try to follow same protocol for collecting data).

The data we are using was collected in part of a "Brain Wide Map" initiative. **You should read the brief overview** [here](https://docs.internationalbrainlab.org/notebooks_external/2025_data_release_brainwidemap.html). Keep in mind of the following:
- The raw data can be obtained from [here](https://docs.internationalbrainlab.org/notebooks_external/data_structure.html).
- The data is organized in a particular format, **which you should read** from [here](https://docs.internationalbrainlab.org/notebooks_external/data_structure.html).
- Data is of various format, but we are primarily interested in `raw_ephys_data` (raw, electrophysiological data colelcted from Neuropixels)

We will use the [ONE API](https://int-brain-lab.github.io/iblenv/notebooks_external/data_download.html) setup to download data.

**Note: The raw data is very large!**

In [ ]:
from one.api import ONE
from pathlib import Path

# Specify download directory (used later)
DOWNLOAD_DIR = Path('IBL_data')
DOWNLOAD_DIR.mkdir(parents = True, exist_ok = True)

one = ONE(
    base_url = 'https://openalyx.internationalbrainlab.org',
    password = 'international',
    cache_dir = DOWNLOAD_DIR,
    silent = True
)

### Querying for Data from Experiment
We will use the data that was also used in [this](https://proceedings.neurips.cc/paper_files/paper/2023/file/83c637c3bc0ca88eda6cf4f5f45bdced-Paper-Conference.pdf) paper. I am quoting a snippet below:
> To train and evaluate our model, we make use of two publicly available extracellular recordings
published by the International Brain Laboratory (IBL): the DY016 and DY009 recordings [54]. These
multi-region, Neuropixels 1.0 recordings are taken from a mouse performing a decision-making task
(see Supplementary Materials Section C for more details).

We will look at the raw MEA data from `DY_016`, which can (also) be found [here](https://ibl.flatironinstitute.org/public/danlab/Subjects/DY_016/2020-09-11/001/raw_ephys_data/). We will load in data from the experiment on date `2019-09-11` session `001`.

You can cross-check by checking the online directory here: https://ibl.flatironinstitute.org/public/danlab/Subjects/DY_016/.

In [ ]:
# Find the exact session (DY_016 / 2020-09-11 / 001)
eids = one.search(
    subject    = 'DY_016',
    date_range = ['2020-09-11', '2020-09-11'],
    number     = 1 # "001" session number
)  

if not eids: raise RuntimeError("Session not found.")

# Convert experiment ids into a list of strings (if not already)
if not isinstance(eids, list):
    eids = [str(eid) for eid in [eids]]

# The triplet (subject, date, session number) corresponds to an experiment.
# This is uniquely identifiable by an experiement id (or eID).
eid = eids[0]

### Number of Probes
Now, every experiment is going to have several different datasets. We only care about the collection with **raw electrophysiological data**. Depending on the number of probes (MEAs), we may have multiple sets of collections.

**Note: The raw data for each probe could be up to 50 GB! While this experiment has two probes (`00` and `01`), we will only download data for the first one!**

This experiment has two probes, as seen here: https://ibl.flatironinstitute.org/public/danlab/Subjects/DY_016/2020-09-11/001/raw_ephys_data/.

In [ ]:
# List all folders that have the format "raw_ephys_data/probe*"
collections = one.list_collections(eid, collection = 'raw_ephys_data/probe*')

print(f'# Probes: {len(collections)}')
print(f'Names: {[col.split("/")[-1] for col in collections]}')

print(collections[-1])

### Relevant Files to Download (from each probe)
For each probe, we want to download the following files (where "ap" means action potential):
- `.ap.meta` (metadata)
- `.ap.cbin` (compressed raw binary)
- `.ap.ch` (commpression header chunks)

**First, we get the relevant file paths and ids.**

Note these file suffixes are slightly different from what you will see in the online directory (but same content, the API just displays the names differetly). For example, see the following for probe `000`: https://ibl.flatironinstitute.org/public/danlab/Subjects/DY_016/2020-09-11/001/raw_ephys_data/probe00/

In [ ]:
import pandas as pd

# This will get datasets for a SINGLE probe (as a data frame)
df = one.list_datasets(eid, collection='raw_ephys_data/probe00', details=True)

# Ensure df is a DataFrame (convert if necessary)
if not isinstance(df, pd.DataFrame): df = pd.DataFrame(df)

# Files we want to download (action potential recordings related)
SUFFIXES = ('.ap.cbin', '.ap.meta', '.ap.ch')

# Keep only the AP files we care about
mask = df['rel_path'].str.endswith(SUFFIXES)
ap_df = df.loc[mask, ['rel_path', 'file_size']].copy()

# Get file size in GB
ap_df['size_GB'] = ap_df['file_size'] / 1e9

# Display only rel_path and size_GB, sorted by rel_path
ap_df[['rel_path', 'size_GB']].sort_values('rel_path')

### All in One Function (Downloading Raw Data)
Now, we will download all the data. Below I provide a useful function for copying/pasting. Just be sure to edit the following (if you know what you are doing):
- `DOWNLOAD_DIR`
- `SUFFIXES`
- `config`

**Note: Unfortunately, we have to download entire file first and then keep the desired time length.**

In [ ]:
# This code block will be useful for copying/pasting to easily download relevant data 
from pathlib import Path
from mtscomp import Reader as CReader, compress as ccompress
from one.api import ONE
import shutil, tempfile
import numpy as np
import shutil

DOWNLOAD_DIR = Path('IBL_data')
DOWNLOAD_DIR.mkdir(parents = True, exist_ok = True)
SUFFIXES = ('.ap.cbin', '.ap.meta', '.ap.ch')

configs: list[dict[str, str | int | list[str]]] = []

# Can add your own
config = {
    'subject':    'DY_016',
    'date_range': ['2020-09-11', '2020-09-11'],
    'number': 1,           # Session "001"
    'length': [0, 5 * 60], # Keep recording from the 0th second up to (and including) 300th second (5 mins).
    'prefix': '5_min',     # Prefix for files to save
    'all probes': False    # If true, downloads all. Else, downloads first one.
}
configs.append(config)


def trim_ibl_cbin_folder(folder: Path, start_s: float, end_s: float, out_prefix: str, keep_originals: bool = False) -> tuple[Path, Path, Path]:
    """
    Create a trimmed SpikeGLX/IBL trio (*.ap.cbin/.ap.ch/.ap.meta) in the same folder.
    Preserves geometry/gains/scale via the meta; no sidecars needed.

    Returns (out_cbin, out_ch, out_meta).
    """
    folder = Path(folder)
    cbin = next(folder.glob("*.ap.cbin"))
    ch   = next(folder.glob("*.ap.ch"))
    meta = next(folder.glob("*.ap.meta"))

    # --- read sample rate & channel count from meta (SpikeGLX-style k=v lines) ---
    print('Reading in data for trimming...')
    md = {}
    for ln in meta.read_text().splitlines():
        if "=" in ln:
            k, v = ln.split("=", 1)
            md[k.strip()] = v.strip()
    fs = float(md.get("imSampRate") or md.get("niSampRate"))  # AP -> imSampRate
    nchan = int(md["nSavedChans"])

    start_f = int(start_s * fs)
    end_f   = int(end_s * fs)
    if end_f <= start_f:
        raise ValueError("end_s must be > start_s")

    # --- stream slice original cbin to a temporary raw int16 .bin ---
    print('Trimming original data...')
    r = CReader(); r.open(str(cbin), str(ch))
    with tempfile.TemporaryDirectory(dir=folder) as td:
        tmp_bin = Path(td) / f"{out_prefix}.ap.bin"
        step = int(1 * fs)
        with open(tmp_bin, "wb") as f:
            s = start_f
            while s < end_f:
                e = min(s + step, end_f)
                arr = r[s:e, :]   # shape (samples, channels), dtype int16
                arr.tofile(f)
                s = e
        r.close()

        # --- recompress to .cbin/.ch (IBL mtscomp) ---
        out_cbin = folder / f"{out_prefix}.ap.cbin"
        out_ch   = folder / f"{out_prefix}.ap.ch"
        ccompress(str(tmp_bin), str(out_cbin), str(out_ch), sample_rate=fs, n_channels=nchan, dtype=np.int16)

    # --- write matching .meta: copy original, patch duration & size ---
    print('Writing new data...')
    out_meta = folder / f"{out_prefix}.ap.meta"
    shutil.copy2(meta, out_meta)
    dur_s = end_s - start_s
    file_size_bytes = (end_f - start_f) * nchan * 2  # int16 -> 2 bytes

    lines = out_meta.read_text().splitlines()
    def set_or_add(key, val):
        for i, ln in enumerate(lines):
            if ln.startswith(key + "="):
                lines[i] = f"{key}={val}"
                return
        lines.append(f"{key}={val}")

    set_or_add("fileTimeSecs", f"{int(dur_s)}")      # integer seconds is sufficient
    set_or_add("fileSizeBytes", f"{file_size_bytes}")
    if not any(ln.startswith("fileSHA1=") for ln in lines):
        lines.append("fileSHA1=0")
    out_meta.write_text("\n".join(lines) + "\n")

    # Optionally remove originals
    if not keep_originals:
        for p in (cbin, ch, meta):
            p.unlink(missing_ok=True)

    return out_cbin, out_ch, out_meta
    
def _move_and_cleanup(file_paths: list[Path], destination_dir: Path, prefix: str, length: list[int] | None = None):
    """
    Move files to destination, optionally trim recording, and clean up.
    
    Args:
        file_paths: List of PosixALFPath objects (or Path-like objects)
        destination_dir: Path object for destination directory
        length: [start_seconds, end_seconds] to extract. If None, keeps entire recording.
    """
    # Ensure destination exists
    destination_dir = Path(destination_dir)
    destination_dir.mkdir(parents = True, exist_ok = True)
    
    # Move each file
    for file_path in file_paths:
        dest_file = destination_dir / file_path.name
        shutil.copy2(file_path, dest_file)
    
    # If length is specified, trim the recording
    if length is not None:
        start_s, end_s = length
        trim_ibl_cbin_folder(destination_dir, start_s, end_s, prefix)

        
    # Delete the downloaded directory from ONE API
    downloaded_root_dir = Path(file_paths[0].parts[0]) / file_paths[0].parts[1]
    shutil.rmtree(downloaded_root_dir)

def _download_raw_ephys_data_helper(one, eid, probe) -> list[Path]:
    files = one.list_datasets(eid, collection = f'raw_ephys_data/{probe}', details = False)
    
    # File to keep
    keep_files = [str(file) for file in files if any([str(file).endswith(suffix) for suffix in SUFFIXES])]
    
    keep_file_paths: list[Path] = []
    for keep_file in keep_files:
        file_path = one.load_dataset(eid, dataset = keep_file, download_only = True)
        keep_file_paths.append(Path(file_path))
    
    return keep_file_paths

def download_raw_ephys_data(configs: list[dict]):
    one = ONE(
        base_url  = 'https://openalyx.internationalbrainlab.org',
        password  = 'international',
        cache_dir = DOWNLOAD_DIR,
        silent    = True
    )
    
    for config in configs:
        # Get all eIDs
        eids = one.search(
            subject    = config['subject'],
            date_range = config['date_range'],
            number     = config['number']
        )
        if not isinstance(eids, list): eids = [str(eid) for eid in [eids]]
        
        for eid in eids:
            # Get all probes
            collections = one.list_collections(eid, collection = 'raw_ephys_data/probe*')
            probes = [col.split("/")[-1] for col in collections]
            
            # Download only first probe (which will be listed last), if specified.
            if not config['all probes']: probes = [probes[-1]]
            
            for probe in probes:
                # Download files
                print(f'Downloading {config["subject"]}_{probe}...')
                file_paths = _download_raw_ephys_data_helper(one, eid, probe)
                
                # Move files, optionally trim, and delete extra ones downloaded
                probe_dir = DOWNLOAD_DIR / f"{config['subject']}_{probe}"
                print(f'Moving files to {probe_dir}...')
                _move_and_cleanup(file_paths, probe_dir, prefix = config.get('prefix', 'trimmed'), length = config.get('length'))
                print('Done!')
    print(f'All files downloaded and processed in {DOWNLOAD_DIR}.')

download_raw_ephys_data(configs)

# Preprocessing (Real World Data)
IBL has a standard preprocessing pipeline outlined [here](https://figshare.com/articles/online_resource/Spike_sorting_pipeline_for_the_International_Brain_Laboratory/19705522?file=49783080). **Most of these steps are done to account for hardware and recording artifcats that happen in experimental settings.**

SpikeInterface calls the technique about "IBL destriping" and have a minimal version of it implemented [here](https://spikeinterface.readthedocs.io/en/stable/modules/preprocessing.html#how-to-implement-ibl-destriping-or-spikeglx-catgt-in-spikeinterface). **This minimal preprocessing pipeline is what we will follow for real recordings.**

Here are the list of steps performed, according to the IBL standard preprocessing pipeline:
> In many cases, we encounter line noise due to voltage leakage on the probe. **This tarnslates into large ”stripes” of noise spanning
the whole probe**. While our first recommendation [...]. **To reduce the impact of these noise ”stripes”, we introduce six main pre-processing steps**:
> 1. High pass filter
> 2. Correction for ”sample shift” along the length of the probe by aligning the samples with a frequency domain approach.
> 3. Automatic detection, rejection and interpolation of failing channels.
> 4. Application of a spatial low-cutfilter.
> 5. Muting of saturated samples. __(NOT APPLIED IN MINIMAL VERSION)__
> 6. ”Whitening”: normalization and decorrelation of the voltage traces via zero component analysis __(NOT APPLIED IN MINIMAL VERSION)__

Another resource from SpikeInterface for analyizing Neuropixel probe data:
- https://spikeinterface.readthedocs.io/en/stable/how_to/analyze_neuropixels.html

In [ ]:
# Some preliminaries
from pathlib import Path

BASE_PATH = Path("IBL_data/DY_016_probe00")

# Create a cached dir (probably won't create when generating datasets...)
CACHE_DIR = Path("IBL_data/cached_DY_016_probe00")
CACHE_DIR.mkdir(parents = True, exist_ok = True)

# Some processes (like drift detection) will take a while, so these settings speed it up
JOB_KWARGS = dict(
    n_jobs         = -1,
    pool_engine    = "process",
    chunk_duration = "1s",
    progress_bar   = True
)

In [ ]:
# Heavy import (imports everything in SpikeInterface, should only use for notebooks)
import spikeinterface.full as si

# IBL uses custom lossless compression, so we use this special fn to read data
raw_rec = si.read_cbin_ibl(folder_path = BASE_PATH)

# Some statistics
sf         = raw_rec.get_sampling_frequency()
num_ch     = raw_rec.get_num_channels()
duration_s = raw_rec.get_total_duration()
print(f"Loaded Data | Freq = {sf:.1f} Hz | Channels = {num_ch} | Duration = {duration_s:.1f} seconds")

# print('Visualization of the probe used (near the insertion end)')
# fig, ax = plt.subplots(figsize=(15, 10))
# si.plot_probe_map(raw_rec, ax=ax, with_channel_ids=True)
# ax.set_ylim(-100, 100)
# plt.show()

## Basic Preprocessing Steps
We implement the preprocessing steps outlined earlier.

**Note: We can save recording after preprocessing, but it is generally not worth it. This step takes minimial time (compared to drift correction) and will still output a huge file.**

In [ ]:
print('Starting Preprocessing')

# ---------------- STEP 1: High-pass ----------------
hp_rec = si.highpass_filter(recording = raw_rec, freq_min = 300.0)

# ---------------- STEP 2: Re-phase (sample-shift correction) ----------------
ps_rec = si.phase_shift(recording = hp_rec)

# ---------------- STEP 3: Interploate Bad Channels ----------------
interp_rec = si.detect_and_interpolate_bad_channels(recording = ps_rec)

# ---------------- STEP 4: Spatial low-cut (destriping) ----------------
hps_rec = si.highpass_spatial_filter(recording = interp_rec)

# Again, we skip Steps 5 and 6 completely!

print('Done Preprocessing')

# Optional (takes long time): save recording
# hps_rec.save(folder = CACHE_DIR / 'preprocess', **JOB_KWARGS)

## Motion / Drift Correction
Neuropixel MEAs are suspectible to drift (ie., animal moving will cause device to move and hence recordings to shift). To correct for this, we will use [DREDGE](https://www.biorxiv.org/content/10.1101/2022.12.04.519043v2) detection. SpikeInterface has a helpful guide: https://spikeinterface.readthedocs.io/en/stable/modules/motion_correction.html#.

Some notes:
- If we want to create hybrid recordings (recordings where we add synthetic neurons), we need to save motion information.
- We should also consider saving motion corrected recording (as this process takes a long time)

In [ ]:
# Note: Saving takes a while
save_motion_data  = True
save_mc_recording = True
mc_folder    = CACHE_DIR / "motion_info" if save_motion_data else None
mc_recording = CACHE_DIR / "motion_corrected_rec" if save_mc_recording else None

# Get motion corrected recording and motion profile (save motion profile, used later for hybrid recording)
mc_rec, motion = si.correct_motion(
    recording     = hps_rec,
    preset        = "dredge",
    output_motion = True,
    folder        = mc_folder,
    **JOB_KWARGS
)

# Save motion corrected recording
# mc_rec.save(folder = mc_recording, **JOB_KWARGS)

## All in One Function (Preprocessing)

SpikeInterface provides support for easy pipelining. Last line is commented out.

**Note: THIS PIPELINE DOES NOT APPLY STEPS 5 and 6 FROM STANDARD IBL PREPROCESSING PIPELINE.**

In [ ]:
from pathlib import Path
import spikeinterface.preprocessing as spre
import spikeinterface.extractors as se

BASE_PATH = Path("IBL_data/DY_016_probe00")

CACHE_DIR = Path("IBL_data/cached_DY_016_probe00")
CACHE_DIR.mkdir(parents = True, exist_ok = True)

# Some processes (like drift detection) will take a while, so these settings speed it up
JOB_KWARGS = dict(
    n_jobs         = -1,
    pool_engine    = "process",
    chunk_duration = "1s",
    progress_bar   = True
)

JOB_KWARGS = dict(
    n_jobs         = -1,
    pool_engine    = "process",
    chunk_duration = "1s",
    progress_bar   = True
)

SYN_PRE_MOTION = {
    "bandpass_filter": {"freq_min": 300.0, "freq_max": 6000.0},
    "astype": {"dtype": "float32"},
}

REAL_PRE_MOTION = {
    'highpass_filter': {"freq_min": 300.0},     # Step 1
    'phase_shift': {},                          # Step 2
    'detect_and_interpolate_bad_channels': {},  # Step 3
    'highpass_spatial_filter': {},              # Step 4
    "astype": {"dtype": "float32"},
}

def preprocess_chain(rec, synthetic = False, correction_motion = True,  save_motion = False, save_rec = False):
    # 0. Figuring out if saving files
    mc_folder        = CACHE_DIR / "motion_info" if save_motion else None
    final_rec_folder = CACHE_DIR / "final_rec" if save_rec else None
    
    # 1. Apply specified preprocessing steps
    pipeline = SYN_PRE_MOTION if synthetic else REAL_PRE_MOTION
    print('Applying pre-motin correction IBL preprocesing...')
    preprocessed_rec = si.apply_preprocessing_pipeline(
        recording        = rec,
        pipeline_or_dict = pipeline
    )

    # 2. Apply drift correction separately
    motion_corrected_rec = preprocessed_rec
    motion               = None
    if correction_motion:
        print('Doing motion correction...')
        motion_corrected_rec, motion = spre.correct_motion(
            recording     = preprocessed_rec,
            preset        = "dredge",
            folder        = mc_folder,
            output_motion = True,
            overwrite     = True,
            **JOB_KWARGS
        )
    
    # 2. Save recording (optional)
    final_rec = motion_corrected_rec
    if save_rec:
        print('Saving final recording...')
        final_rec.save(folder = final_rec_folder, **JOB_KWARGS)
    
    return final_rec, motion

print("Reading Raw Data")
raw_rec = se.read_cbin_ibl(folder_path=BASE_PATH)
print("Raw Recording Specs\n", raw_rec)

print("Starting preprocessing")
# Returns fully preprocessed recording along with motion data
final_rec, motion = preprocess_chain(raw_rec, save_motion=True, save_rec=True)
print("Final Recording Specs\n", final_rec)
print("Done preprocessing!")

# Generating Synthetic Data
The following documentation will be useful: https://spikeinterface.readthedocs.io/en/stable/modules/generation.html.

Here are the key ideas:
- **Synethic recordings**: raw generated recordings (based on specified MEA device). We can specify ground truth (number of neruons, or "units") and many other details (like drift settings / profiles).

- **Hybrid recordings**: add synthetic units to recordings. Useful for robustness to noise.

- **Noise recordings**: add Gaussian noise to recordings.

In [1]:
# Defiing some preliminiaries (again, just in case)
# Some preliminaries
from pathlib import Path

BASE_PATH = Path("IBL_data/DY_016_probe00")

# Create a cached dir (only for demonstration purposes)
CACHE_DIR = Path("IBL_data/cached_DY_016_probe00")
CACHE_DIR.mkdir(parents = True, exist_ok = True)

# Some processes (like drift detection) will take a while, so these settings speed it up
JOB_KWARGS = dict(
    n_jobs         = -1,
    pool_engine    = "process",
    chunk_duration = "1s",
    progress_bar   = True
)

## Synthetic Recordings
Below is how we generate a synthetic recording with drift. We get the following as output:
- Static recording (no drift)
- Drift recording
- Ground truth information

Some notes:
- We may want to utilize synthetic recordings with and without drift (for evaluation purposes).
- We may to play around with the kwargs of `generate_drifting_recording(...)` to get various drift settings.
- We will not need to do **most** of the preprocesing steps (most of it is only applicable to real data)

In [ ]:
# This is a heavy import, but imports everything from SpikeInterface
import spikeinterface.full as si

# This will generate a single recording (300 seconds in length).
static_rec, drift_rec, gt_sorting = si.generate_drifting_recording(
    probe_name         = "Neuropixels1-384", # Neuropixel 1.0 w/ 384 channels (matching the one used in DY_016)
    num_units          = 200,                # Neurons
    duration           = 300,                # Length of recording (seconds)
    sampling_frequency = 30000,              # In Hz
    seed               = 4776
)

### Synthetic Recordings: Difference Between `static_rec` and `drift_rec`
The `generate_drifting_recording()` function returns two recordings that contain the **same neurons and spike trains**, but differ in how they simulate the recording conditions:

#### `static_rec` (Static Recording)
- **No drift/motion**: The positions of neurons relative to the recording electrodes remain **constant** throughout the recording
- **Ideal conditions**: Represents a perfect, stable recording environment
- **Use cases**:
  - Baseline comparisons (what's the best possible performance?)
  - Testing algorithms under ideal conditions
  - Understanding algorithm behavior without motion artifacts
  - Ground truth reference

#### `drift_rec` (Drift Recording)
- **Simulates drift/motion**: The relative positions of neurons and electrodes **change over time**
- **Realistic conditions**: Mimics real-world scenarios where:
  - The animal moves (causing the brain to shift relative to the probe)
  - The probe can move slightly during long recordings
  - Brain tissue can shift due to physiological processes
- **Drift patterns**: Can follow various motion profiles (zigzag, bumps, random walk, etc.)
- **Use cases**:
  - Realistic benchmarking (how does the algorithm handle motion?)
  - Testing motion correction algorithms
  - Evaluating spike sorting robustness to drift
  - Better simulation of real experimental conditions

#### Key Differences:

| Feature | `static_rec` | `drift_rec` |
|---------|-------------|-------------|
| Neuron positions | Fixed relative to electrodes | Change over time |
| Motion artifacts | None | Simulated drift patterns |
| Realism | Ideal conditions | Realistic conditions |
| Spike waveforms | Constant shape per neuron | Shape changes as neurons move |
| Clustering difficulty | Easier (more stable) | Harder (more challenging) |
| Use for evaluation | Baseline performance | Realistic performance |

#### Why Both?
Having both recordings allows you to:
1. **Compare performance**: See how much drift affects your spike sorting algorithm
2. **Benchmark algorithms**: Test under ideal (static) vs. realistic (drift) conditions
3. **Validate motion correction**: Test if motion correction can recover static-like performance
4. **Evaluate robustness**: Understand how sensitive your methods are to drift

## Hybrid Recordings
Below is how we generate hybrid recordings. We will get the following:
- Hybrid recordings (with injected new units)
- Ground truth information (about the injected units)

Here is what we need for this:
1. A real recording
2. The recording must be preprocessed (**except with unwhitening**)
3. If there is drift / motion in the recording, we need the motion profile. The recording itself SHOULD NOT be drift corrected.

Some notes:
- We should paly around with the kwargs of `generate_hybrid_recording(...)`.

In [ ]:
# If this is not true, we are assuming "motion" variable exist already.
# This is motion of recording after all preprocessing (except unwhitening).
loading_from_folder = True
if loading_from_folder:
    motion_info = si.load_motion_info(CACHE_DIR / 'motion_info')
    motion      = motion_info['motion']

# We use our function from the previous section to get the non-drift corrected version but pre-processed.
preprocess_rec, _ = preprocess_chain(
    rec               = si.read_cbin_ibl(BASE_PATH),
    correction_motion = False,
    apply_post_motion = False, # Want un-whitened
)

# It should be the case that "motion" is the motion for the "preprocess_rec." We do it like this b/c
# getting motion takes long time, and we already .

# Again, it would be worth exploring the kwargs to generate variety of recordings
hybrid_rec, gt_sorting = si.generate_hybrid_recording(
    recording  = preprocess_rec,
    motion     = motion, # Add motion present in original recording to synthetic units being added
    seed       = 4776
)

# Should be 10, can change with generate_sorting_kwargs kwarg in generate_hybrid_recording(...)
print('# Units Added:', gt_sorting.get_num_units())

## Noise Recordings (Under Construction)
We take real recordings and inject Gaussian noise.

In [ ]:
# In progress

# Getting Matrix (Extracting Waveforms)
We need to understand the **sampling rate** and **traces** before we can perform any dimension reduction.

## Understanding Sampling Rate

Neuropixels and other MEA typically have a **sample rate** of 30,000 Hertz (Hz) or (30kHz). In other words, there are 30,000 measurements / samples of voltage recorded for every channel on the MEA. MEAs record at such a fine resolution because neuron spikes are very quick events (1-3 ms) and can be easily missed for less finer resolutoins.

For example, for a given recording length, we get the following the number of samples:
- 1 second = 30,000 samples
- 2 seconds = 60,000 samples
- 10 seconds = 300,000 samples
- 300 seconds = 9,000,000 samples

## Understanding Traces

**Traces** are the voltage signals recorded over time from each electrode (channel). **This is the raw data we want to turn into a matrix.**

Each trace represents the voltage signal from one electrode over time. When a neuron fires near an electrode, it creates a characteristic voltage "spike" that appears in the trace as a brief deflection.

For a Neuropixels1-384 recording with 384 channels, each trace has the same number of time samples. In our 300-second recording example:
- 300 seconds × 30,000 Hz = 9,000,000 samples per trace
- Shape as numpy array: `(9,000,000 samples, 384 channels)`
- Each **row** = all channels at one time point
- Each **column** = one channel's trace over all time

The traces contain the actual spikes we're looking for (brief 1-3 ms voltage deflections when neurons fire), along with baseline noise from background electrical activity and occasional artifacts from motion or line noise.

We extract traces to:
- **Spike detection**: Find when neurons fire
- **Clustering**: Group spikes from the same neuron
- **Dimensionality reduction**: Simplify the 384-channel space
- **Feature extraction**: Characterize spike shapes

Below is some code to take a recording and output a numpy array.

In [ ]:
# Convert recording to numpy array for clustering and dimensionality reduction
import numpy as np

def recording_to_array(recording, start_frame=None, end_frame=None, save_path=None, verbose=True):
    """
    Convert a SpikeInterface recording to a numpy array.
    
    Parameters:
    -----------
    recording : BaseRecording
        A SpikeInterface recording object (e.g., from si.generate_drifting_recording())
    start_frame : int, optional
        Start frame index (inclusive). If None, starts from beginning.
    end_frame : int, optional
        End frame index (exclusive). If None, goes to end.
    save_path : str or Path, optional
        If provided, save the numpy array to this path (e.g., 'data/synthetic_recording.npy').
        Will create parent directories if they don't exist.
    verbose : bool, default=True
        If True, print information about the extracted array.
    
    Returns:
    --------
    numpy.ndarray
        Array of shape (num_samples, num_channels) where:
        - Each row = one time point (all channels at that moment)
        - Each column = one channel's trace over all time
    
    Examples:
    ---------
    >>> # Extract entire recording
    >>> synthetic_data = recording_to_array(drift_rec)
    
    >>> # Extract first 1 second and save to disk
    >>> subset = recording_to_array(drift_rec, start_frame=0, end_frame=30000,
    ...                             save_path='data/1second_recording.npy')
    
    >>> # Extract and save without verbose output
    >>> data = recording_to_array(drift_rec, save_path='full_recording.npy', verbose=False)
    """
    from pathlib import Path
    
    # Extract traces from the recording
    if start_frame is not None or end_frame is not None:
        traces = recording.get_traces(start_frame=start_frame, end_frame=end_frame)
    else:
        traces = recording.get_traces()
    
    # Print information if requested
    if verbose:
        print(f"Recording shape: {traces.shape}")
        print(f"Data type: {traces.dtype}")
        print(f"Memory size: {traces.nbytes / 1e9:.2f} GB")
        print(f"\nNumber of samples: {traces.shape[0]:,}")
        print(f"Number of channels: {traces.shape[1]}")
    
    # Save to disk if path provided
    if save_path is not None:
        save_path = Path(save_path)
        # Create parent directories if they don't exist
        save_path.parent.mkdir(parents=True, exist_ok=True)
        # Ensure .npy extension
        if save_path.suffix != '.npy':
            save_path = save_path.with_suffix('.npy')
        
        np.save(save_path, traces)
        if verbose:
            print(f"\nSaved to: {save_path.absolute()}")
            print(f"File size: {save_path.stat().st_size / 1e9:.2f} GB")
    
    return traces

# Example usage: Convert the drift recording to numpy array
synthetic_data = recording_to_array(drift_rec)

# Example: Convert and save to disk
# synthetic_data = recording_to_array(drift_rec, save_path='data/synthetic_drift_recording.npy')

# Example: Convert static recording (uncomment to use)
# static_data = recording_to_array(static_rec)
# static_data = recording_to_array(static_rec, save_path='data/synthetic_static_recording.npy')

# Example: Extract a subset and save (first 10 seconds at 30kHz = 300,000 samples)
# subset_data = recording_to_array(drift_rec, start_frame=0, end_frame=300000, save_path='data/subset_10seconds.npy')

# Download Real World Data (Under Construction)
The recordings are still actually very large. See what I mean below:

In [ ]:
from pathlib import Path
import spikeinterface.full as si

BASE_PATH = Path("/IBL_data/cached_DY_016_probe00")
final_rec = si.load(BASE_PATH / 'final_rec')

print(final_rec)